In [1]:
# STEP 1: Install packages
!pip install -q chromadb sentence-transformers rank-bm25 anthropic


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.0/833.0 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [2]:
# STEP 2: Import libraries
import json
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from typing import List, Dict, Any, Optional


In [3]:
# STEP 3: Load chunks (need this for BM25 and metadata)
from google.colab import files
uploaded = files.upload()  # Upload processed_chunks.json

with open('./processed_chunks.json', 'r') as f:
    chunks = json.load(f)
print(f"✅ Loaded {len(chunks)} chunks")



Saving processed_chunks.json to processed_chunks.json
✅ Loaded 187 chunks


In [4]:
# STEP 4: Upload and extract database
from google.colab import files
uploaded = files.upload()  # Upload chroma_db_export.zip

import shutil
shutil.unpack_archive("my_meeting_notes_db.zip", "./chroma_db")
print("✅ Database extracted")



Saving my_meeting_notes_db.zip to my_meeting_notes_db.zip
✅ Database extracted


In [5]:
# STEP 5: Initialize embedding model (need for new queries)
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded


In [6]:
# STEP 6: Connect to existing ChromaDB
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_collection(name="gbac_meetings")
print(f"✅ Connected! Documents: {collection.count()}")


✅ Connected! Documents: 187


In [7]:
# STEP 7: Define search functions
def semantic_search(query, n_results=5, date_filter=None):
    """Perform semantic search using vector similarity."""
    query_embedding = model.encode(query)
    where_filter = None
    if date_filter:
        where_filter = {"meeting_date": date_filter}

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=n_results,
        where=where_filter
    )

    formatted = []
    for i in range(len(results['ids'][0])):
        formatted.append({
            'content': results['documents'][0][i],
            'metadata': results['metadatas'][0][i],
            'similarity_score': 1 - results['distances'][0][i]
        })
    return formatted

def display_results(results, max_length=300):
    """Display search results in a nice format."""
    print(f"\n{'='*80}")
    print(f"Found {len(results)} results")
    print(f"{'='*80}\n")

    for i, result in enumerate(results, 1):
        score = result.get('similarity_score', 0)
        metadata = result.get('metadata', {})

        print(f"Result {i} | Score: {score:.3f}")
        print("-" * 80)
        print(f"📁 File: {metadata.get('filename', 'Unknown')}")
        if 'meeting_date' in metadata:
            print(f"📅 Date: {metadata['meeting_date']}")
        print(f"📌 Section: {metadata.get('section_type', 'Unknown')}")

        content = result['content']
        if len(content) > max_length:
            content = content[:max_length] + "..."
        print(f"\n{content}\n")

print("✅ Search functions defined")


✅ Search functions defined


In [8]:
# STEP 8: Build BM25 index (need to rebuild from chunks)
tokenized_corpus = [chunk.get('text', '').lower().split() for chunk in chunks]
bm25 = BM25Okapi(tokenized_corpus)

def keyword_search(query, n_results=5):
    """Perform keyword-based BM25 search."""
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:n_results]

    results = []
    for idx in top_indices:
        chunk = chunks[idx]
        chunk_meta = chunk.get('metadata', {})
        results.append({
            'content': chunk.get('text', ''),
            'metadata': {
                'filename': chunk_meta.get('filename', 'unknown'),
                'section_type': chunk_meta.get('section_type', chunk_meta.get('section', 'unknown')),
                'meeting_date': chunk_meta.get('meeting_date', ''),
            },
            'bm25_score': float(scores[idx])
        })
    return results

print("✅ BM25 index built")


✅ BM25 index built


In [9]:
# STEP 9: Define hybrid search
def hybrid_search(query, n_results=5, semantic_weight=0.6):
    """Combines semantic and keyword search."""
    semantic_results = semantic_search(query, n_results=n_results*2)
    keyword_results = keyword_search(query, n_results=n_results*2)

    rrf_scores = {}
    k = 60

    for rank, result in enumerate(semantic_results, 1):
        key = result['content'][:100]
        rrf_scores[key] = {
            'score': semantic_weight / (k + rank),
            'result': result
        }

    for rank, result in enumerate(keyword_results, 1):
        key = result['content'][:100]
        if key in rrf_scores:
            rrf_scores[key]['score'] += (1 - semantic_weight) / (k + rank)
        else:
            rrf_scores[key] = {
                'score': (1 - semantic_weight) / (k + rank),
                'result': result
            }

    sorted_results = sorted(rrf_scores.values(), key=lambda x: x['score'], reverse=True)

    final_results = []
    for item in sorted_results[:n_results]:
        result = item['result']
        result['hybrid_score'] = item['score']
        final_results.append(result)

    return final_results

print("✅ Hybrid search defined")



✅ Hybrid search defined


In [10]:
# STEP 10: Create QueryInterface
class QueryInterface:
    """Easy interface for searching meeting notes"""

    def search(self, query, method='hybrid', n_results=5, date_filter=None):
        """
        Main search function - one interface for all methods

        Args:
            query: Your search question
            method: 'semantic', 'keyword', or 'hybrid' (default)
            n_results: Number of results (default 5)
            date_filter: Optional date like "October 2, 2024"
        """
        print(f"🔍 Searching: '{query}'")
        print(f"   Method: {method}")
        if date_filter:
            print(f"   Date filter: {date_filter}")
        print("="*80 + "\n")

        # Choose search method
        if method == 'semantic':
            results = semantic_search(query, n_results, date_filter)
        elif method == 'keyword':
            results = keyword_search(query, n_results)
            # Apply date filter manually for keyword search
            if date_filter:
                results = [r for r in results
                          if r['metadata'].get('meeting_date') == date_filter]
        else:  # hybrid
            results = hybrid_search(query, n_results)
            # Apply date filter manually for hybrid search
            if date_filter:
                results = [r for r in results
                          if r['metadata'].get('meeting_date') == date_filter]

        # Display
        display_results(results)
        return results

    def find_by_section(self, section_query, n_results=10, exact_match=False):
        """
        Find all chunks from a specific section type.
        Now supports both exact and flexible matching!

        Args:
            section_query: Section name (e.g., "Discussion" or "Green Building Discussion")
            n_results: Number of results to return
            exact_match: If True, requires exact match. If False, allows partial matches.
        """
        section_query_lower = section_query.lower()

        if exact_match:
            print(f"📑 Finding sections exactly matching '{section_query}'")
        else:
            print(f"📑 Finding sections containing '{section_query}'")
        print("="*80 + "\n")

        results = []
        for chunk in chunks:
            chunk_meta = chunk.get('metadata', {})
            section_type = chunk_meta.get('section_type') or chunk_meta.get('section', '')

            # Check match based on mode
            is_match = False
            if exact_match:
                is_match = (section_type == section_query)
            else:
                is_match = (section_query_lower in section_type.lower())

            if is_match:
                results.append({
                    'content': chunk.get('text', ''),
                    'metadata': {
                        'filename': chunk_meta.get('filename', 'unknown'),
                        'section_type': section_type,
                        'meeting_date': chunk_meta.get('meeting_date', ''),
                    },
                    'similarity_score': 1.0
                })
                if len(results) >= n_results:
                    break

        # Show helpful message if no results
        if not results:
            print(f"⚠️  No sections found matching '{section_query}'")
            print(f"\n💡 Available sections:")
            available = set()
            for chunk in chunks:
                chunk_meta = chunk.get('metadata', {})
                section = chunk_meta.get('section_type') or chunk_meta.get('section')
                if section:
                    available.add(section)
            for s in sorted(available):
                print(f"  • {s}")
            print(f"\n💡 Try: qi.find_by_section('discussion') for partial match")
            return []

        display_results(results)
        return results

    def find_by_date_range(self, query, start_date, end_date, n_results=10):
        """
        Search within a date range.

        Args:
            query: Search query
            start_date: Start date (e.g., "February 1, 2024")
            end_date: End date (e.g., "December 31, 2024")
            n_results: Number of results
        """
        print(f"🔍 Searching: '{query}'")
        print(f"   Date range: {start_date} to {end_date}")
        print("="*80 + "\n")

        # Get all results first
        all_results = hybrid_search(query, n_results=n_results*3)

        # Filter by date range
        from datetime import datetime

        filtered = []
        for result in all_results:
            meeting_date = result['metadata'].get('meeting_date', '')
            if meeting_date:
                try:
                    # Try to parse the date
                    result_date = datetime.strptime(meeting_date, "%B %d, %Y")
                    start_dt = datetime.strptime(start_date, "%B %d, %Y")
                    end_dt = datetime.strptime(end_date, "%B %d, %Y")

                    if start_dt <= result_date <= end_dt:
                        filtered.append(result)
                except:
                    pass

        filtered = filtered[:n_results]

        if not filtered:
            print(f"⚠️  No results found in date range")
        else:
            display_results(filtered)

        return filtered

    def find_recent(self, query, n_meetings=3, n_results_per_meeting=2):
        """
        Search recent meetings only.

        Args:
            query: Search query
            n_meetings: Number of recent meetings to search
            n_results_per_meeting: Results per meeting
        """
        # Get all unique dates without printing
        dates_set = set()
        for chunk in chunks:
            date = chunk.get('metadata', {}).get('meeting_date')
            if date:
                dates_set.add(date)

        # Sort descending (most recent first)
        recent_dates = sorted(dates_set, reverse=True)[:n_meetings]

        print(f"🔍 Searching '{query}' in {n_meetings} most recent meetings")
        print(f"   Dates: {', '.join(recent_dates)}")
        print("="*80 + "\n")

        all_results = []

        for date in recent_dates:
            print(f"📅 {date}")
            print("-"*80)

            # Call semantic_search directly with date filter
            date_results = semantic_search(
                query,
                n_results=n_results_per_meeting,
                date_filter=date
            )

            if date_results:
                # Display compact version
                for i, result in enumerate(date_results, 1):
                    score = result.get('similarity_score', 0)
                    section = result['metadata'].get('section_type', 'Unknown')
                    filename = result['metadata'].get('filename', 'Unknown')
                    content_preview = result['content'][:120] + "..."

                    print(f"\n  Result {i} | Score: {score:.3f}")
                    print(f"  📁 {filename}")
                    print(f"  📌 {section}")
                    print(f"  {content_preview}")

                all_results.extend(date_results)
            else:
                print("  ⚠️  No relevant results found for this date")

            print()  # Blank line between dates

        print("="*80)
        print(f"✅ Total: {len(all_results)} results across {n_meetings} meetings\n")

        return all_results

    def list_all_dates(self):
        """Show all available meeting dates"""
        dates = set()
        for chunk in chunks:
            date = chunk.get('metadata', {}).get('meeting_date')
            if date:
                dates.add(date)

        print("📅 Available Meeting Dates:")
        print("="*80)
        for date in sorted(dates):
            print(f"  • {date}")
        print(f"\nTotal: {len(dates)} meetings")
        return sorted(dates)

    def list_all_sections(self):
        """Show all section types in your data"""
        sections = set()
        for chunk in chunks:
            chunk_meta = chunk.get('metadata', {})
            section = chunk_meta.get('section_type') or chunk_meta.get('section')
            if section:
                sections.add(section)

        print("📑 Available Section Types:")
        print("="*80)
        for section in sorted(sections):
            print(f"  • {section}")
        print(f"\nTotal: {len(sections)} section types")
        return sorted(sections)

    def get_meeting_summary(self, date):
        """
        Get all sections from a specific meeting date.

        Args:
            date: Meeting date (e.g., "October 2, 2024")
        """
        print(f"📋 Meeting Summary for {date}")
        print("="*80 + "\n")

        meeting_chunks = []
        for chunk in chunks:
            chunk_meta = chunk.get('metadata', {})
            if chunk_meta.get('meeting_date') == date:
                section = chunk_meta.get('section_type') or chunk_meta.get('section', 'Unknown')
                meeting_chunks.append({
                    'section': section,
                    'content': chunk.get('text', '')[:200] + "..."
                })

        if not meeting_chunks:
            print(f"⚠️  No data found for {date}")
            return []

        # Group by section
        sections_found = {}
        for chunk in meeting_chunks:
            section = chunk['section']
            if section not in sections_found:
                sections_found[section] = []
            sections_found[section].append(chunk['content'])

        print(f"Found {len(meeting_chunks)} chunks in {len(sections_found)} sections:\n")
        for section in sorted(sections_found.keys()):
            print(f"📌 {section}")
            print(f"   {len(sections_found[section])} chunks")

        print(f"\n💡 Try: qi.search('your question', date_filter='{date}')")
        return meeting_chunks


# Create the interface
qi = QueryInterface()

print("✅ Enhanced Query Interface created!")
print("\n📚 Available methods:")
print("  • qi.search(query, method='hybrid', n_results=5, date_filter=None)")
print("  • qi.find_by_section(section_query, n_results=10, exact_match=False)")
print("  • qi.find_by_date_range(query, start_date, end_date, n_results=10)")
print("  • qi.find_recent(query, n_meetings=3, n_results_per_meeting=2)")
print("  • qi.get_meeting_summary(date)")
print("  • qi.list_all_dates()")
print("  • qi.list_all_sections()")
print("\n💡 Example features:")
print("  ✨ Flexible section search (partial matching)")
print("  ✨ Date range queries")
print("  ✨ Search recent meetings only")
print("  ✨ Get complete meeting summaries")

✅ Enhanced Query Interface created!

📚 Available methods:
  • qi.search(query, method='hybrid', n_results=5, date_filter=None)
  • qi.find_by_section(section_query, n_results=10, exact_match=False)
  • qi.find_by_date_range(query, start_date, end_date, n_results=10)
  • qi.find_recent(query, n_meetings=3, n_results_per_meeting=2)
  • qi.get_meeting_summary(date)
  • qi.list_all_dates()
  • qi.list_all_sections()

💡 Example features:
  ✨ Flexible section search (partial matching)
  ✨ Date range queries
  ✨ Search recent meetings only
  ✨ Get complete meeting summaries


In [11]:
# STEP 11: RAG System - AI-Powered Q&A over Meeting Notes (OpenAI)

class RAGSystem:
    """
    Retrieval-Augmented Generation system for meeting notes.
    Combines vector search with LLM to answer questions.
    """

    def __init__(self, query_interface, api_key=None, provider='openai'):
        """
        Initialize RAG system.

        Args:
            query_interface: Your qi object from Step 13
            api_key: API key for LLM provider (optional, will prompt if needed)
            provider: 'anthropic' or 'openai'
        """
        self.qi = query_interface
        self.provider = provider
        self.api_key = api_key

        # Will be set when API key is provided
        self.client = None

        print(f"✅ RAG System initialized with {provider}")
        if not api_key:
            print("⚠️  No API key provided. Use set_api_key() before asking questions.")

    def set_api_key(self, api_key, provider=None):
        """Set API key for LLM provider"""
        self.api_key = api_key
        if provider:
            self.provider = provider

        # Initialize client based on provider
        if self.provider == 'anthropic':
            try:
                import anthropic
                self.client = anthropic.Anthropic(api_key=api_key)
                print("✅ Anthropic API connected (Claude)")
            except ImportError:
                print("❌ Install anthropic: pip install anthropic")
                return False
        elif self.provider == 'openai':
            try:
                import openai
                self.client = openai.OpenAI(api_key=api_key)
                print("✅ OpenAI API connected (GPT)")
            except ImportError:
                print("❌ Install openai: pip install openai")
                return False

        return True

    def ask(
        self,
        question,
        n_results=5,
        search_method='hybrid',
        date_filter=None,
        show_sources=True,
        temperature=0.3
    ):
        """
        Ask a question and get an AI-generated answer with citations.

        Args:
            question: Your question
            n_results: Number of context chunks to retrieve
            search_method: 'semantic', 'keyword', or 'hybrid'
            date_filter: Optional date filter
            show_sources: Whether to show source documents
            temperature: LLM temperature (0-1, lower = more focused)
        """
        if not self.client:
            print("❌ API key not set! Run: rag.set_api_key('your-key')")
            return None

        print(f"🔍 Searching for relevant context...")
        print(f"   Question: {question}")
        if date_filter:
            print(f"   Date filter: {date_filter}")
        print()

        # Step 1: Retrieve relevant chunks
        if search_method == 'semantic':
            results = semantic_search(question, n_results, date_filter)
        elif search_method == 'keyword':
            results = keyword_search(question, n_results)
            if date_filter:
                results = [r for r in results if r['metadata'].get('meeting_date') == date_filter]
        else:  # hybrid
            results = hybrid_search(question, n_results)
            if date_filter:
                results = [r for r in results if r['metadata'].get('meeting_date') == date_filter]

        if not results:
            print("❌ No relevant context found for this question.")
            return None

        print(f"✓ Found {len(results)} relevant chunks\n")

        # Step 2: Build context from results
        context = self._build_context(results)

        # Step 3: Generate answer with LLM
        print(f"🤖 Generating answer with {self.provider}...")
        answer = self._generate_answer(question, context, temperature)

        # Step 4: Display results
        print("\n" + "=" * 80)
        print("💬 ANSWER:")
        print("=" * 80)
        print(f"\n{answer}\n")

        if show_sources:
            self._display_sources(results)

        return {
            'question': question,
            'answer': answer,
            'sources': results,
            'context_used': context
        }

    def _build_context(self, results):
        """Build context string from search results"""
        context_parts = []

        for i, result in enumerate(results, 1):
            metadata = result['metadata']
            filename = metadata.get('filename', 'Unknown')
            date = metadata.get('meeting_date', 'Unknown date')
            section = metadata.get('section_type', 'Unknown section')
            content = result['content']

            context_parts.append(
                f"[Source {i}]\n"
                f"File: {filename}\n"
                f"Date: {date}\n"
                f"Section: {section}\n"
                f"Content: {content}\n"
            )

        return "\n---\n".join(context_parts)

    def _generate_answer(self, question, context, temperature):
        """Generate answer using LLM"""

        # Create prompt
        prompt = f"""You are an expert analyst reviewing DC Department of Energy & Environment's Green Building Advisory Council (GBAC) meeting notes.

Based on the following meeting notes excerpts, answer the user's question. Be specific and cite which meetings or sections support your answer.

Context from meeting notes:
{context}

User's question: {question}

Instructions:
- Provide a clear, concise answer based only on the context provided
- If the context mentions specific dates, people, or projects, include them
- If the context doesn't fully answer the question, say what information is available and what's missing
- Cite which meetings (by date) support your statements
- If there are conflicting views or ongoing discussions, mention that

Answer:"""

        # Call appropriate LLM
        if self.provider == 'anthropic':
            return self._call_anthropic(prompt, temperature)
        elif self.provider == 'openai':
            return self._call_openai(prompt, temperature)

    def _call_anthropic(self, prompt, temperature):
        """Call Anthropic Claude API"""
        try:
            message = self.client.messages.create(
                model="claude-3-5-sonnet-20241022",
                max_tokens=1024,
                temperature=temperature,
                messages=[
                    {"role": "user", "content": prompt}
                ]
            )
            return message.content[0].text
        except Exception as e:
            return f"Error calling Anthropic API: {str(e)}"

    def _call_openai(self, prompt, temperature):
        """Call OpenAI GPT API"""
        try:
            response = self.client.chat.completions.create(
                model="gpt-4o",  # Using GPT-4o (latest model)
                messages=[
                    {"role": "system", "content": "You are an expert analyst of government meeting notes."},
                    {"role": "user", "content": prompt}
                ],
                temperature=temperature,
                max_tokens=1024
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Error calling OpenAI API: {str(e)}"

    def _display_sources(self, results):
        """Display source documents"""
        print("=" * 80)
        print("📚 SOURCES:")
        print("=" * 80)

        for i, result in enumerate(results, 1):
            metadata = result['metadata']
            score = result.get('similarity_score') or result.get('hybrid_score') or result.get('bm25_score', 0)

            print(f"\n[Source {i}] Relevance: {score:.3f}")
            print(f"  📁 File: {metadata.get('filename', 'Unknown')}")
            print(f"  📅 Date: {metadata.get('meeting_date', 'Unknown')}")
            print(f"  📌 Section: {metadata.get('section_type', 'Unknown')}")
            print(f"  Preview: {result['content'][:150]}...")

    def summarize_meeting(self, date, focus=None):
        """
        Generate an AI summary of a specific meeting.

        Args:
            date: Meeting date (e.g., "October 2, 2024")
            focus: Optional focus area (e.g., "action items", "decisions")
        """
        if not self.client:
            print("❌ API key not set!")
            return None

        print(f"📋 Generating summary for meeting: {date}")
        if focus:
            print(f"   Focus: {focus}")
        print()

        # Get all chunks from this meeting
        meeting_chunks = []
        for chunk in chunks:
            chunk_meta = chunk.get('metadata', {})
            if chunk_meta.get('meeting_date') == date:
                meeting_chunks.append(chunk)

        if not meeting_chunks:
            print(f"❌ No data found for meeting: {date}")
            return None

        print(f"✓ Found {len(meeting_chunks)} chunks from this meeting\n")

        # Build context
        context_parts = []
        for chunk in meeting_chunks:
            chunk_meta = chunk.get('metadata', {})
            section = chunk_meta.get('section_type', chunk_meta.get('section', 'Unknown'))
            content = chunk.get('text', '')
            context_parts.append(f"[{section}]\n{content}")

        context = "\n\n".join(context_parts)

        # Create summary prompt
        if focus:
            focus_instruction = f"Focus especially on: {focus}"
        else:
            focus_instruction = "Cover all major topics discussed."

        prompt = f"""Summarize this GBAC meeting from {date}.

{focus_instruction}

Meeting content:
{context}

Provide a structured summary with:
1. Main topics discussed
2. Key decisions or recommendations
3. Action items (if any)
4. Notable concerns or questions raised

Summary:"""

        print(f"🤖 Generating summary...")
        summary = self._generate_answer("Summarize meeting", context, temperature=0.3)

        print("\n" + "=" * 80)
        print(f"📋 MEETING SUMMARY - {date}")
        print("=" * 80)
        print(f"\n{summary}\n")

        return summary

    def compare_meetings(self, dates, topic):
        """
        Compare how a topic evolved across multiple meetings.

        Args:
            dates: List of meeting dates
            topic: Topic to track
        """
        if not self.client:
            print("❌ API key not set!")
            return None

        print(f"📊 Comparing topic across meetings: {topic}")
        print(f"   Dates: {', '.join(dates)}\n")

        # Get relevant chunks from each meeting
        meeting_contexts = []
        for date in dates:
            results = semantic_search(topic, n_results=3, date_filter=date)
            if results:
                context = f"Meeting {date}:\n"
                context += "\n".join([r['content'] for r in results])
                meeting_contexts.append(context)

        if not meeting_contexts:
            print("❌ No relevant information found across these meetings")
            return None

        combined_context = "\n\n---\n\n".join(meeting_contexts)

        prompt = f"""Analyze how the topic of "{topic}" evolved across these GBAC meetings.

Meeting excerpts:
{combined_context}

Provide:
1. How the discussion of this topic changed over time
2. Key developments or shifts in approach
3. Recurring themes or concerns
4. Overall trajectory

Analysis:"""

        print(f"🤖 Generating comparison...")
        analysis = self._generate_answer(f"Compare {topic} across meetings", combined_context, temperature=0.3)

        print("\n" + "=" * 80)
        print(f"📊 TOPIC EVOLUTION: {topic}")
        print("=" * 80)
        print(f"\n{analysis}\n")

        return analysis


# Create RAG system
rag = RAGSystem(qi, provider='openai')

print("✅ RAG System created!")
print("\n📚 Available methods:")
print("  • rag.set_api_key('your-key')                       - Set up OpenAI API")
print("  • rag.ask('your question')                           - Ask questions")
print("  • rag.summarize_meeting('October 2, 2024')           - Summarize meeting")
print("  • rag.compare_meetings([dates], 'topic')             - Track topic evolution")
print("\n⚠️  API Key Source:")
print("   Get OpenAI key: https://platform.openai.com/api-keys")
print("\n💡 Start with: rag.set_api_key('your-api-key-here')")

✅ RAG System initialized with openai
⚠️  No API key provided. Use set_api_key() before asking questions.
✅ RAG System created!

📚 Available methods:
  • rag.set_api_key('your-key')                       - Set up OpenAI API
  • rag.ask('your question')                           - Ask questions
  • rag.summarize_meeting('October 2, 2024')           - Summarize meeting
  • rag.compare_meetings([dates], 'topic')             - Track topic evolution

⚠️  API Key Source:
   Get OpenAI key: https://platform.openai.com/api-keys

💡 Start with: rag.set_api_key('your-api-key-here')


In [12]:
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')

In [13]:
# STEP 12: Set API key

rag.set_api_key(api_key, provider="openai")

# Ready to use!
print("\n" + "="*80)
print("🎉 SYSTEM READY!")
print("="*80)
print(f"Database: {collection.count()} documents loaded")
print("Try: rag.ask('your question here')")

✅ OpenAI API connected (GPT)

🎉 SYSTEM READY!
Database: 187 documents loaded
Try: rag.ask('your question here')


In [14]:
# ============================================================
# EXAMPLE 1: Ask a Question
# ============================================================

print("📝 EXAMPLE 1: Ask a Question")
print("=" * 80 + "\n")

# Ask about energy efficiency requirements
result = rag.ask(
    question="What energy efficiency requirements were discussed for new buildings?",
    n_results=5,
    search_method='hybrid'
)

print("\n\n")


# ============================================================
# EXAMPLE 2: Question with Date Filter
# ============================================================

print("📝 EXAMPLE 2: Question About Specific Meeting")
print("=" * 80 + "\n")

result = rag.ask(
    question="What were the main topics discussed?",
    date_filter="October 2, 2024",
    n_results=5
)

print("\n\n")


# ============================================================
# EXAMPLE 3: Summarize a Meeting
# ============================================================

print("📝 EXAMPLE 3: Meeting Summary")
print("=" * 80 + "\n")

summary = rag.summarize_meeting(
    date="October 2, 2024"
)

print("\n\n")


# ============================================================
# EXAMPLE 4: Focused Summary
# ============================================================

print("📝 EXAMPLE 4: Focused Meeting Summary")
print("=" * 80 + "\n")

summary = rag.summarize_meeting(
    date="October 2, 2024",
    focus="action items and decisions"
)

print("\n\n")


# ============================================================
# EXAMPLE 5: Compare Across Meetings
# ============================================================

print("📝 EXAMPLE 5: Track Topic Evolution")
print("=" * 80 + "\n")

analysis = rag.compare_meetings(
    dates=["August 7, 2024", "October 2, 2024", "December 4, 2024"],
    topic="building code amendments"
)

📝 EXAMPLE 1: Ask a Question

🔍 Searching for relevant context...
   Question: What energy efficiency requirements were discussed for new buildings?

✓ Found 5 relevant chunks

🤖 Generating answer with openai...

💬 ANSWER:

The energy efficiency requirements discussed for new buildings in the context of the Green Building Advisory Council (GBAC) meetings include several key areas:

1. **Decarbonization and Energy Efficiency**: The GBAC recommended reviewing the ASHRAE/ASHE Decarbonizing Hospital Buildings Guidebook to identify strategies for improving energy efficiency and thermal energy performance. They also suggested investigating opportunities to use heat pumps to minimize or eliminate the use of electric resistance boilers (April 2, 2025, meeting).

2. **Life Cycle Cost Analysis (LCCA)**: The GBAC recommended conducting an LCCA that accounts for utility cost escalation to compare electric resistance equipment with a combination of heat pumps and electric resistance. This analysis s

In [15]:
# ============================================================
# Complex Policy Analysis
# ============================================================

# Multi-part question
rag.ask("""
What is the Green Building Act's position on renewable energy requirements,
and how has the council's interpretation evolved over 2024?
""", n_results=8)


# ============================================================
# Stakeholder Tracking
# ============================================================

# Find what specific organizations said
rag.ask(
    "What concerns did DGS raise about the exemption process?",
    n_results=5
)


# ============================================================
# Action Item Tracking
# ============================================================

# Find all action items
rag.ask(
    "What action items were assigned in recent meetings and who is responsible?",
    n_results=10
)


# ============================================================
# Comparative Analysis
# ============================================================

# Compare approaches
rag.compare_meetings(
    dates=["February 5, 2025", "April 2, 2025"],
    topic="LEED certification requirements"
)

🔍 Searching for relevant context...
   Question: 
What is the Green Building Act's position on renewable energy requirements,
and how has the council's interpretation evolved over 2024?


✓ Found 8 relevant chunks

🤖 Generating answer with openai...

💬 ANSWER:

The Green Building Act's position on renewable energy requirements involves the procurement of off-site renewable energy via power purchase agreements (PPAs) or bundled Renewable Energy Certificates (RECs) from Tier 1 renewable energy sources. These sources must meet the minimum percentage of the District’s Renewable Portfolio Standard and be geographically limited to the PJM interconnection region. This was specified in the GBAC meeting on May 15, 2024 (Source 2).

Throughout 2024, the council's interpretation and application of these requirements have evolved, as evidenced by discussions during the May 15, 2024 meeting. The GBAC expressed concerns about placing the responsibility of procuring renewable energy solely on non-pro

'The meeting notes provided do not explicitly discuss LEED certification requirements. Instead, they focus on compliance with the Greener Government Buildings Amendment Act (GGBA) and related energy performance metrics for specific projects, such as the Congress Heights Recreation Center (CHRC) and Howard University Hospital.\n\nFrom the February 5, 2025, meeting, the CHRC project initially sought exemptions from GGBA requirements but ultimately redesigned to comply with the law, focusing on eliminating natural gas-fired heating and opting for all-electric systems (February 5, 2025, meeting notes).\n\nIn the April 2, 2025, meeting, Howard University Hospital sought exemptions from various GGBA requirements, including energy use intensity and thermal energy performance. The GBAC recommended partial, conditional exemptions for renewable energy requirements but did not vote on exemptions related to building energy use intensity and thermal energy performance, suggesting the project was to

In [16]:
# ============================================================
# ADVANCED ATTENDEE QUERIES
# ============================================================

print("👥 ADVANCED ATTENDEE ANALYSIS EXAMPLES")
print("="*80 + "\n")

# Example 1: Track Individual Participation
print("📊 EXAMPLE 1: Individual Member Participation")
print("-"*80)
result = rag.ask(
    question="Which GBAC members attended the most meetings in 2024? Track their participation patterns.",
    n_results=15,
    show_sources=True
)
print("\n\n")


# Example 2: Organizational Representation
print("📊 EXAMPLE 2: Organizational Representation")
print("-"*80)
result = rag.ask(
    question="Which organizations (DOEE, DGS, DOB, HOK, etc.) were consistently represented at meetings throughout 2024 and 2025?",
    n_results=15,
    show_sources=True
)
print("\n\n")


# Example 3: Attendance Trends Over Time
print("📊 EXAMPLE 3: Attendance Trends")
print("-"*80)
result = rag.ask(
    question="How did meeting attendance change between 2023, 2024, and 2025? Were there any patterns in participation?",
    n_results=15,
    show_sources=True
)
print("\n\n")


# Example 4: Specific Member Tracking
print("📊 EXAMPLE 4: Track Specific Member")
print("-"*80)
result = rag.ask(
    question="When did Jenn Hatch from DOEE attend meetings, and what were her key contributions or statements?",
    n_results=15,
    show_sources=True
)
print("\n\n")


# Example 5: Guest/Other Attendees Analysis
print("📊 EXAMPLE 5: Non-Member Attendance")
print("-"*80)
result = rag.ask(
    question="Who were the frequent guest attendees or other participants (not official GBAC members) across meetings in 2024?",
    n_results=15,
    show_sources=True
)
print("\n\n")


# Example 6: Absent Members
print("📊 EXAMPLE 6: Absence Patterns")
print("-"*80)
result = rag.ask(
    question="Which GBAC members were frequently absent from meetings in 2024? Are there any patterns?",
    n_results=15,
    show_sources=True
)
print("\n\n")


# Example 7: Cross-Reference with Topics
print("📊 EXAMPLE 7: Attendees + Topics")
print("-"*80)
result = rag.ask(
    question="Which GBAC members or attendees were present when building code amendments were discussed? Who participated in those discussions?",
    n_results=15,
    show_sources=True
)
print("\n\n")


# Example 8: Quorum Analysis
print("📊 EXAMPLE 8: Meeting Quorum")
print("-"*80)
result = rag.ask(
    question="Which meetings had the highest and lowest attendance? Was there sufficient quorum for decision-making?",
    n_results=15,
    show_sources=True
)
print("\n\n")


# Example 9: New vs. Returning Members
print("📊 EXAMPLE 9: Member Turnover")
print("-"*80)
result = rag.ask(
    question="Were there any new GBAC members who joined in 2024 or 2025? Did any long-standing members leave?",
    n_results=15,
    show_sources=True
)
print("\n\n")


# Example 10: Department-Specific Participation
print("📊 EXAMPLE 10: Department Participation")
print("-"*80)
result = rag.ask(
    question="How frequently did representatives from DGS (Department of General Services) attend and participate in meetings?",
    n_results=15,
    show_sources=True
)

👥 ADVANCED ATTENDEE ANALYSIS EXAMPLES

📊 EXAMPLE 1: Individual Member Participation
--------------------------------------------------------------------------------
🔍 Searching for relevant context...
   Question: Which GBAC members attended the most meetings in 2024? Track their participation patterns.

✓ Found 15 relevant chunks

🤖 Generating answer with openai...

💬 ANSWER:

Based on the provided meeting notes, the GBAC members who attended the most meetings in 2024 are:

1. **Jenn Hatch (DOEE)**: Attended the meetings on February 7, 2024 ([Source 12]), August 7, 2024 ([Source 4]), and August 28, 2024 ([Source 14]).

2. **Stephen Gyor (OP)**: Attended the meetings on February 7, 2024 ([Source 12]), August 7, 2024 ([Source 4]), and August 28, 2024 ([Source 14]).

3. **Linda Toth (Arup)**: Attended the meetings on February 7, 2024 ([Source 12]), August 7, 2024 ([Source 4]), and August 28, 2024 ([Source 14]).

These members were present in all three documented meetings in 2024, indicat

In [17]:
# ============================================================
# DEMO: Interactive Colab Widget
# ============================================================

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# Create demo interface
print("="*80)
print("🎓 GBAC MEETING NOTES SEARCH - PROJECT DEMO")
print("="*80)
print("\nBy: Maurice Onyonyi | Georgetown University")
print("Course: DSPP | Date: January 2026\n")

# Question input widget
question_input = widgets.Textarea(
    value='',
    placeholder='Example: What energy efficiency requirements were discussed?',
    description='Question:',
    layout=widgets.Layout(width='80%', height='100px')
)

# Settings
n_results_slider = widgets.IntSlider(
    value=15,
    min=5,
    max=30,
    step=5,
    description='Results:',
    style={'description_width': 'initial'}
)

search_method = widgets.Dropdown(
    options=['hybrid', 'semantic', 'keyword'],
    value='hybrid',
    description='Method:',
)

show_sources_check = widgets.Checkbox(
    value=True,
    description='Show Sources'
)

# Search button
search_button = widgets.Button(
    description='🔍 Search',
    button_style='primary',
    layout=widgets.Layout(width='150px', height='40px')
)

# Output area
output = widgets.Output()

# Example questions
examples = widgets.Dropdown(
    options=[
        'Custom Question...',
        'What energy efficiency requirements were discussed?',
        'Who attended the most meetings in 2024?',
        'What building projects were reviewed?',
        'How did policies evolve from 2023 to 2025?',
        'What exemptions were requested?'
    ],
    description='Examples:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

def on_example_change(change):
    if change['new'] != 'Custom Question...':
        question_input.value = change['new']

examples.observe(on_example_change, names='value')

def on_search_click(b):
    with output:
        clear_output()

        if not question_input.value.strip():
            print("⚠️ Please enter a question!")
            return

        print("🔍 Searching...\n")

        try:
            result = rag.ask(
                question=question_input.value,
                n_results=n_results_slider.value,
                search_method=search_method.value,
                show_sources=show_sources_check.value,
                temperature=0.3
            )

        except Exception as e:
            print(f"❌ Error: {e}")
            print("\n💡 Make sure:")
            print("  • You've run all setup steps (Steps 1-18)")
            print("  • Your OpenAI API key is set")
            print("  • The database is initialized")

search_button.on_click(on_search_click)

# Display interface
display(HTML("<h3>📝 Ask a Question:</h3>"))
display(examples)
display(question_input)

display(HTML("<h3>⚙️ Settings:</h3>"))
display(widgets.HBox([search_method, n_results_slider, show_sources_check]))

display(search_button)

display(HTML("<h3>📊 Results:</h3>"))
display(output)

print("\n" + "="*80)
print("💡 TIP: Click 'Examples' dropdown to try pre-written questions!")
print("="*80)

🎓 GBAC MEETING NOTES SEARCH - PROJECT DEMO

By: Maurice Onyonyi | Georgetown University
Course: DSPP | Date: January 2026



Dropdown(description='Examples:', layout=Layout(width='80%'), options=('Custom Question...', 'What energy effi…

Textarea(value='', description='Question:', layout=Layout(height='100px', width='80%'), placeholder='Example: …

Button(button_style='primary', description='🔍 Search', layout=Layout(height='40px', width='150px'), style=Butt…

Output()


💡 TIP: Click 'Examples' dropdown to try pre-written questions!
